# Cleaning the country groups dataset

This notebook inspects and cleans the official Latvian migration dataset `ibe010-migration-by-country-groups-2016-2025-raw.csv`.
The goal is to inspect the raw data, convert the wide format year columns into a long format dataset, clean the dataset, validate the result after melting and cleaning, and save the dataset in `data/interim` folder for subsequent transformation.

In [1]:
from pathlib import Path
import pandas as pd

region_groups_data = pd.read_csv("../data/raw/ibe010-migration-by-country-groups-2016-2025-raw.csv", skiprows=2)

region_groups_data.head()

,Rādītāji,Valstu grupa,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Iebrauca Latvijā,Pavisam,8345,9916,10909,11223,8840,12689,38708,15629,14701,13826
1,Iebrauca Latvijā,ES-27 no 2020. g.,2254,2356,2294,2531,2506,3696,4080,4316,3992,3349
2,Iebrauca Latvijā,ES-28,4780,4841,4488,4727,4469,6202,6852,7122,6830,2146
3,Iebrauca Latvijā,ES kandidātvalstis,32,77,106,88,74,176,24857,2739,3061,3335
4,Iebrauca Latvijā,EBTA,185,179,246,254,222,364,423,502,499,364


The required libraries are imported and the raw country groups dataset is loaded.
The first two rows of the `.csv` file contain the dataset title and a blank line before the actual column names, therefore, `skiprows=2` is used so that pandas reads the correct header row.
The first rows confirm that the dataset was loaded with the expected identifier columns, `Rādītāji` and `Valstu grupa`, and the year columns from 2016 to 2025.

In [2]:
region_groups_data.shape

(14, 12)

The dataset contains 14 rows and 12 columns.
The rows represent two migration indicators for seven country groups. The columns consist of two identifier columns, `Rādītāji` and `Valstu grupa`, and ten year columns from 2016 to 2025.
The shape, together with the displayed first rows, confirms that the dataset is structured in a wide format.

In [3]:
region_groups_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Rādītāji      14 non-null     str  
 1   Valstu grupa  14 non-null     str  
 2   2016          14 non-null     int64
 3   2017          14 non-null     int64
 4   2018          14 non-null     int64
 5   2019          14 non-null     int64
 6   2020          14 non-null     int64
 7   2021          14 non-null     int64
 8   2022          14 non-null     int64
 9   2023          14 non-null     int64
 10  2024          14 non-null     int64
 11  2025          14 non-null     int64
dtypes: int64(10), str(2)
memory usage: 1.4 KB


The dataset contains 14 non-null entries for every column.
`Rādītāji` and `Valstu grupa` use pandas default `str` data type. The ten your columns use `int64` data type.
This confirms that all required year columns are present, all values in the year columns are numeric, and there are no missing values in the dataset.

In [4]:
identifier_columns = ["Rādītāji", "Valstu grupa"]

for column in identifier_columns:
    print(region_groups_data[column].value_counts(dropna=False))

Rādītāji
Iebrauca Latvijā        7
Izbrauca no Latvijas    7
Name: count, dtype: int64
Valstu grupa
Pavisam               2
ES-27 no 2020. g.     2
ES-28                 2
ES kandidātvalstis    2
EBTA                  2
NVS                   2
Pārējās valstis       2
Name: count, dtype: int64


The identifier columns contain only the expected values:
- `Rādītāji` contains two migration identifiers: `Iebrauca Latvijā` and `Izbrauca no Latvijas`. Each appears seven times, once for each country group.
- `Valstu grupa` contains seven country groups. Each appears twice: once for arrivals and once for departures.

In [5]:
print("Count of missing values:")
print(region_groups_data.isna().sum())

print("Count of empty strings:")
print(region_groups_data[identifier_columns].eq("").sum())

Count of missing values:
Rādītāji        0
Valstu grupa    0
2016            0
2017            0
2018            0
2019            0
2020            0
2021            0
2022            0
2023            0
2024            0
2025            0
dtype: int64
Count of empty strings:
Rādītāji        0
Valstu grupa    0
dtype: int64


The dataset does not contain missing values in any column. The identifier columns do not contain empty strings either.
This confirms that no missing value handling or empty string handling is required.

In [6]:
for column in identifier_columns:
    stripped_values = region_groups_data[column].astype("string").str.strip()

    whitespace_count = stripped_values.ne(region_groups_data[column].astype("string")).sum()

    print(f"{column}: {whitespace_count} values with leading or trailing whitespace.")

Rādītāji: 0 values with leading or trailing whitespace.
Valstu grupa: 0 values with leading or trailing whitespace.


The identifier columns are checked for leading or trailing whitespace, which may not be visible when the values are displayed.
For each identifier column, the values are stripped of leading and trailing whitespace and compared with the original values.
Bot identifier columns contain no leading or trailing whitespace, therefore, no whitespace cleaning is required.

In [7]:
region_groups_data[identifier_columns].duplicated().sum()

np.int64(0)

The identifier columns are checked for duplicate entries in the dataset.
Result of `0` confirms that no duplicates are present. No duplicate rows need to be removed.

In [8]:
year_columns = region_groups_data.columns[2:]

region_groups_melted = region_groups_data.melt(
    id_vars=identifier_columns,
    value_vars=year_columns,
    var_name="Gads",
    value_name="Skaits"
)

region_groups_melted.head()

,Rādītāji,Valstu grupa,Gads,Skaits
0,Iebrauca Latvijā,Pavisam,2016,8345
1,Iebrauca Latvijā,ES-27 no 2020. g.,2016,2254
2,Iebrauca Latvijā,ES-28,2016,4780
3,Iebrauca Latvijā,ES kandidātvalstis,2016,32
4,Iebrauca Latvijā,EBTA,2016,185


The dataset is converted from wide format to long format.
`year_columns` selects the ten year columns. The `.melt()` method keeps the identifier columns, and converts each year column into a row. The former year column names are stored in `Gads`, and their published migration counts are stored in `Skaits`.
The resulting dataset contains one row for every migration indicator, country group, and year.

In [9]:
region_groups_melted["Gads"] = region_groups_melted["Gads"].astype("int64")

After melting, `Gads` column contains former year column names. During melting, these column names are stored as strings, so `Gads` has the `str` data type.
`Gads` is converted to `int64`, matching the data type of the year columns in the original dataset. This keeps the years as numeric, ordered values for later transformations, such as filtering or comparing values by year, if necessary.

In [10]:
region_groups_melted.shape

(140, 4)

The melted dataset contains 140 rows and 4 columns.
The rows represent every combination of two migrations indicators, seven country groups, and ten years.
The four columns now are `Rādītāji`, `Valstu grupa`, `Gads`, and `Skaits`.
This confirms that the melting from wide to long format has been successful.

In [11]:
region_groups_melted.info()

<class 'pandas.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Rādītāji      140 non-null    str  
 1   Valstu grupa  140 non-null    str  
 2   Gads          140 non-null    int64
 3   Skaits        140 non-null    int64
dtypes: int64(2), str(2)
memory usage: 4.5 KB


The melted dataset, just as the original, contains no null values in its 140 rows.
`Rādītāji` and `Valstu grupa` keep `str` data type as intended. `Gads` and `Skaits` use `int64` data type.
This confirms that the year conversion has been successful, and that `Gads` conversion has also succeeded.

In [12]:
print("All counts are non-negative:")
print((region_groups_melted["Skaits"] >= 0).all())

print("All counts are whole numbers:")
print((region_groups_melted["Skaits"] % 1 == 0).all())

All counts are non-negative:
True
All counts are whole numbers:
True


The cleaned migration counts are checked to confirm that they are valid. `Skaits >= 0` checks that every published count is non-negative. `Skaits % 1 == 0` checks that every count is a whole number.
Both checks return `True`, confirming that no invalid values were introduced by accident during cleaning.

In [13]:
checks_to_compare = [("Iebrauca Latvijā", "Pavisam", 2016, 8345),
                     ("Iebrauca Latvijā", "Pavisam", 2025, 13826),
                     ("Izbrauca no Latvijas", "Pavisam", 2016, 20574),
                     ("Izbrauca no Latvijas", "Pavisam", 2025, 15117)
]

for indicator, country_group, year, expected_count in checks_to_compare:
    actual_count = region_groups_melted.loc[
        (region_groups_melted["Rādītāji"] == indicator)
        & (region_groups_melted["Valstu grupa"] == country_group)
        & (region_groups_melted["Gads"] == year),
        "Skaits"
    ].item()

    print(f"{indicator} | {country_group} | {year}: expected {expected_count}, found {actual_count}")

Iebrauca Latvijā | Pavisam | 2016: expected 8345, found 8345
Iebrauca Latvijā | Pavisam | 2025: expected 13826, found 13826
Izbrauca no Latvijas | Pavisam | 2016: expected 20574, found 20574
Izbrauca no Latvijas | Pavisam | 2025: expected 15117, found 15117


Selected rows are checked to confirm that they still match the correct migration flow, country group, and year after melting.
For each check, the expected count is taken from the originally loaded dataset. The melted dataset is filtered by migration indicator, country group, and year, and the resulting `Skaits` value is compared with the expected count.
All selected values match, confirming that the melting preserved the correct migration flows, country groups, years and published counts.

In [14]:
region_groups_melted.to_csv("../data/interim/ibe010-migration-by-country-groups-2016-2025-interim.csv", index=False, encoding="utf-8-sig")
print("The dataset has been saved")

The dataset has been saved


The cleaned long format dataset is saved to the `data/interim` folder for use in transformations as the next step in the planned methodology.
`index=False` prevents pandas from writing its numeric row index as an extra column in the `.csv` file. `encoding="utf-8-sig"` writes a UTF-8 file with a bite order mark, which helps some programs, including Microsoft Excel, to correctly recognize the encoding and display Latvian characters.